In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import

In [2]:
!pip install sentence_transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 33.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 33.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 92.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.2 MB/s eta 0:00:00
  Created wheel for sentence_transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=42f6f4e97c3a4bcc2ad3736a29258288fe08625a2d521e8a8e3f0b5cf2d5c3ab
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence_transformers


In [3]:
import re
import pandas as pd
import numpy as np
import random
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

import warnings
warnings.filterwarnings("ignore")

## Random Seed

In [4]:
SEED = 0

np.random.seed(SEED)
random.seed(SEED)

## Load Data

In [5]:
df = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/news.csv')
# df.head()

In [6]:
# 제목 + 내용
df['text'] = df['title'] + ' : ' + df['contents']
# df.head()

## Pre-processing

In [7]:
def preprocess_text(text):
    #1 URL 제거
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    #2 문자열 ";p", "/p"를 모두 삭제
    text = re.sub(r'[;/]p', '', text)

    #3 HREF로 시작하는 패턴 모두 삭제
    text = re.sub(r'HREF="[^"]*"', '', text)

    # #4 &과 이어진 문장 모두 삭제
    # text = re.sub(r'&[^;]+;', '', text)

    #5 '/'와 '\' 문자 모두 삭제
    text = re.sub(r'[\/\\]', '', text)

    #6 해시태그 제거
    text = re.sub(r'#\w+', '', text)

    #7 멘션 제거
    text = re.sub(r'@\w+', '', text)

    #8 이모지 제거
    text = text.encode('ascii', 'ignore').decode('ascii')

    #9 공백 및 특수문자 제거
    text = re.sub(r'\s+', ' ', text).strip()

    #10 숫자 제거
    text = re.sub(r'\d+', '', text)

    return text

In [8]:
df['processed_text'] = df['text'].apply(preprocess_text)

## Feature Extraction

In [9]:
# Sentence BERT 모델 로드
model = SentenceTransformer('all-mpnet-base-v2')

# 텍스트 feature 추출
sentence_embeddings = model.encode(df['processed_text'].tolist(), show_progress_bar=True)

from sklearn.preprocessing import MinMaxScaler

# 임베딩을 표준화
scaler = MinMaxScaler()
scaled_embeddings = scaler.fit_transform(sentence_embeddings)

# 추출한 feature를 데이터프레임에 저장
df_embeddings = pd.DataFrame(scaled_embeddings)

Batches:   0%|          | 0/1875 [00:00<?, ?it/s]

## Clustering

In [10]:
# Sentence BERT 임베딩을 사용하여 군집화 수행
kmeans = KMeans(n_clusters=6, random_state=SEED)

df['kmeans_cluster'] = kmeans.fit_predict(scaled_embeddings)

## Post-processing

### Tech: 0 -> 4

In [11]:
df[df['kmeans_cluster'] == 0]['text']

3        Macromedia contributes to eBay Stores : Macrom...
4        Qualcomm plans to phone it in on cellular repa...
5        Thomson to Back Both Blu-ray and HD-DVD : Comp...
23       FTC Files First Lawsuit Against Spyware Concer...
31       Sony PSP Draws Crowds and Lines on First Day (...
                               ...                        
59972    SBC Details Fiber Plans : SBC Communications I...
59975    Supreme Court Won #39;t Weigh Net Music Lawsui...
59976    The Sims Go to College : November 22, 2004 - I...
59978    SMIC to challenge latest TSMC infringement cla...
59983    Partners Weigh In On Firefox, IE Faceoff : Sol...
Name: text, Length: 9285, dtype: object

### Politics: 1 -> 2

In [12]:
df[df['kmeans_cluster'] == 1]['text']

8        Obama Marks Anniversary Of 9/11 Attacks With M...
9        Republican Congressman Says Trump Should Apolo...
11       Kerry rolls out tax-cut plan for middle class ...
12       Read Live Updates From The South Carolina Demo...
14       Obama Administration Helps Wall Street Crimina...
                               ...                        
59951    HUFFPOST HILL - Trump Campaign Pivots... Into ...
59953    A Dozen Lessons Donald Trump Could Learn From ...
59954    You Don't Have To Agree With Donald Trump To B...
59966    Mitt Romney Lambasts Donald Trump As A 'Phony'...
59988    Obama To Call For More Icebreakers In The Arct...
Name: text, Length: 9802, dtype: object

### Sports: 2 -> 3

In [13]:
df[df['kmeans_cluster'] == 2]['text']

0        Spanish coach facing action in race row : MADR...
6        Time to Talk Baseball : It's time to talk abou...
13       GAME DAY PREVIEW Game time: 6:00 PM : CHARLOTT...
16       Fischer's Fiancee: Marriage Plans Genuine (AP)...
21       Blake Leeper Wants to Be the First American Pa...
                               ...                        
59990    Wizards Down Galaxy : Kansas City secures the ...
59991    Irish talk of softening schedule a little : wh...
59993    Olson: Hoping to preserve the joy of Sox : Sin...
59995    Dolphins Break Through, Rip Rams For First Win...
59997    Pro football: Culpepper puts on a show : To sa...
Name: text, Length: 11775, dtype: object

### Business: 3 -> 0

In [14]:
df[df['kmeans_cluster'] == 3]['text']

7        Bump Stock Maker Resumes Sales One Month After...
19       Congress Spikes Handout For Private Equity aut...
20       Deere's Color Is Green : With big tractors, bi...
27       Kmart-Sears merger about price, quality : Aver...
37       Deep Impact Space Probe Aims to Slam Into Come...
                               ...                        
59985    European Shares Tread Water (Reuters) : Reuter...
59986    Higher trade growth predicted in 2004 despite ...
59994    Chain Store Sales Rise in the Latest Week : NE...
59996    After Steep Drop, Price of Oil Rises : The fre...
59998    Albertsons on the Rebound : The No. 2 grocer r...
Name: text, Length: 11706, dtype: object

### Entertainment: 4 -> 1

In [15]:
df[df['kmeans_cluster'] == 4]['text']

2        Only Lovers Left Alive's Tilda Swinton Talks A...
25       Be on TOP : //www.huffingtonpost.com/entry/be-...
28       Cate Blanchett Set To Star As Lucille Ball In ...
40       Out for V-I-C-T-O-R-Y, but Missing Tiles : Mis...
45       The Trouble with Broadcasting in a Social Worl...
                               ...                        
59956    6 Ways to Improve Your Performance at Work aut...
59964    Russell Crowe Reaps Shocking Sum In Divorce Au...
59967    Europeans in Thrall of American Culture (AP) :...
59969    'American Hustle' '12 Years a Slave' Lead 2014...
59981    Khloe Kardashian Gets A Hilarious Lesson In Po...
Name: text, Length: 7753, dtype: object

### World: 5 -> 5

In [16]:
df[df['kmeans_cluster'] == 5]['text']

1        Bruce Lee statue for divided city : In Bosnia,...
10       Harry #39;s argy-bargy : PRINCE Charles has as...
29       Israel Kills 3 Palestinians in Big Gaza Incurs...
34       The Folly of the Sole Superpower Writ Small au...
56       Sadr #39;s aide denies entering of Iraqi polic...
                               ...                        
59982    Militants kill 12 in J amp;K ahead of PMs visi...
59984    The Plot Thickens: Testing European Tolerance ...
59987    Nepal Seeks to Break Rebel Siege with Air Patr...
59992    US troops on offensive in Iraq : US troops wen...
59999    Cassini Craft Spies Saturn Moon Dione (AP) : A...
Name: text, Length: 9679, dtype: object

### Mapping

In [17]:
mapping_dict = {
    0: 4,
    1: 2,
    2: 3,
    3: 0,
    4: 1,
    5: 5
}

In [18]:
df['mapping'] = df['kmeans_cluster'].apply(lambda x: mapping_dict[x])

## Submission

In [19]:
sample = pd.read_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/sample_submission.csv')

In [20]:
sample['category'] = df['mapping'].values
sample['category'].head()

0    3
1    5
2    1
3    4
4    4
Name: category, dtype: int64

In [21]:
sample.to_csv('/content/drive/MyDrive/월간 데이콘 쇼츠 - 뉴스 기사 레이블 복구 해커톤/all-mpnet-base-v2_13.csv', index=False)